## 获得 git message

In [ ]:
import json
from utils import execute_with_info as execute  # 返回 dict 或 None

def get_message_info(json_file_path):
    try:
        # 读取 JSON 文件
        with open(json_file_path, 'r', encoding='utf-8') as file:
            json_data = json.load(file)
        
        # 只处理前 10 条有效数据
        processed_count = 0
        for entry in json_data:
            # if processed_count >= 10:
            #     break

            repo_url = entry.get("repo", "")
            commit_hash = entry.get("commitHash", "")
            
            if not repo_url or not commit_hash:
                continue  # 跳过无效条目

            path = "YOUR_DATA_DIR/gitRepo/" + repo_url.split("/")[-1]  # 修正：取最后一部分（仓库名）

            if not path or not commit_hash:
                continue

            # 调用 execute —— 现在它返回结构化 info 或 None
            result = execute(path, commit_hash)

            if result is not None:
                entry["a_info"] = result["parent1"]      # 包含 message + time
                entry["b_info"] = result["parent2"]
                entry["o_info"] = result["common_ancestor"]
                processed_count += 1
                print(f'{processed_count}: Successfully processed {commit_hash}')
                print(f'{result["parent1"], result["parent2"], result["common_ancestor"]}')
            else:
                # 可选：记录失败，或留空
                entry["a_info"] = None
                entry["b_info"] = None
                entry["o_info"] = None
                print(f'{processed_count + 1}: Failed to process {commit_hash}')

        # 将更新后的数据写回文件
        with open(json_file_path, 'w', encoding='utf-8') as file:
            json.dump(json_data, file, indent=4, ensure_ascii=False)
        
        print(f"Processed {processed_count} entries and saved to {json_file_path}")
    
    except FileNotFoundError:
        print(f"The file {json_file_path} does not exist.")
    except json.JSONDecodeError:
        print(f"Failed to decode JSON from the file {json_file_path}.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

In [ ]:
get_message_info("YOUR_PROJECT_DIR/dataset/mergebert/json_ts/autorest_typescript_copy.json")
get_message_info("YOUR_PROJECT_DIR/dataset/mergebert/json_ts/ssr_copy.json")
get_message_info("YOUR_PROJECT_DIR/dataset/mergebert/json_ts/stratos_copy.json")
get_message_info("YOUR_PROJECT_DIR/dataset/mergebert/json_ts/tv-automation-server-core_copy.json")

# mergeBert预处理（基于时间历史）

## 冲突块获得最迟时间

In [ ]:
import json
from datetime import datetime

def get_latest_commit_time(input_file, output_file):
    # 读取输入文件
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # 处理每个对象
    processed_data = []
    for item in data:
        # 提取时间
        times = []
        if 'a_info' in item and 'time' in item['a_info']:
            time_str = item['a_info']['time'].replace('Z', '+00:00')  # 将 Z 替换为 +00:00
            times.append(datetime.fromisoformat(time_str))
        if 'b_info' in item and 'time' in item['b_info']:
            time_str = item['b_info']['time'].replace('Z', '+00:00')
            times.append(datetime.fromisoformat(time_str))
        if 'o_info' in item and 'time' in item['o_info']:
            time_str = item['o_info']['time'].replace('Z', '+00:00')
            times.append(datetime.fromisoformat(time_str))
        
        # 找出最迟时间
        if times:
            latest_time = max(times).isoformat()
        else:
            latest_time = None
        
        # 创建新对象，移除a_info, b_info, o_info，添加time
        new_item = {k: v for k, v in item.items() if k not in ['a_info', 'b_info', 'o_info']}
        new_item['timestamp'] = latest_time
        processed_data.append(new_item)

    # 写入输出文件
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(processed_data, f, indent=4, ensure_ascii=False)

    print("处理完成，输出文件已生成。")

In [ ]:
get_latest_commit_time("YOUR_PROJECT_DIR/dataset/mergebert/json_ts/autorest_typescript_copy.json",
                       "YOUR_PROJECT_DIR/dataset/mergebert/json_ts/autorest_typescript_time.json")
get_latest_commit_time("YOUR_PROJECT_DIR/dataset/mergebert/json_ts/ssr_copy.json",
                       "YOUR_PROJECT_DIR/dataset/mergebert/json_ts/ssr_time.json")
get_latest_commit_time("YOUR_PROJECT_DIR/dataset/mergebert/json_ts/stratos_copy.json",
                       "YOUR_PROJECT_DIR/dataset/mergebert/json_ts/stratos_time.json")
get_latest_commit_time("YOUR_PROJECT_DIR/dataset/mergebert/json_ts/tv-automation-server-core_copy.json",
                       "YOUR_PROJECT_DIR/dataset/mergebert/json_ts/tv-automation-server-core_time.json")

## 时间json——vector

In [41]:
import json

def json_to_vector(input_path, output_path):
    with open(input_path, 'r', encoding='utf-8') as file:
        data_list = json.load(file)

    valid_records_count = 0

    with open(output_path, 'w', encoding='utf-8') as output_file:
        for data in data_list:
            res_label = data.get('res_label', '')
            if res_label == 'null':
                continue
            a_contents = data.get('a_contents', '')
            o_contents = data.get('o_contents', '')
            b_contents = data.get('b_contents', '')
            res_region = data.get('res_region', '')
            global_id = data.get('id', '')
            timestamp = data.get('timestamp', '')
            
            conflict = 'concurrent_change\n' + a_contents + '\nbase_content\n' + o_contents + '\nincoming_change\n' + b_contents
            processed_conflict = conflict.replace('\n', ' _newline_ ').replace('\t', ' ')
            processed_res = res_region.replace('\n', ' _newline_ ').replace('\t', ' ')
            output_file.write(processed_conflict + '\t' + processed_res + '\t' + str(global_id) + '\t' + timestamp + '\n')
            valid_records_count += 1

    print(f"处理完毕，结果已写入到文件中。共处理了 {valid_records_count} 条有效记录。")

In [42]:
json_to_vector(input_path = "dataset/after/mergebert_20/mergebert_20.json", output_path = 'dataset/mergebert/vector/mergebert_20_Conflicts.txt')

处理完毕，结果已写入到文件中。共处理了 1673 条有效记录。


## 根据时间进行排序

In [20]:
import json
from datetime import datetime

def sort_by_time(input_path):
    def parse_timestamp(ts: str):
        if not ts:
            return None
        try:
            return datetime.fromisoformat(ts)
        except ValueError:
            return None


    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # 按时间从晚到早排序；缺失或非法时间排在末尾
    sorted_data = sorted(
        data,
        key=lambda item: parse_timestamp(item.get("timestamp", "")) or datetime.min,
        reverse=True,
    )

    with open(input_path, "w", encoding="utf-8") as f:
        json.dump(sorted_data, f, indent=2, ensure_ascii=False)

    print(f"已按时间从晚到早排序并写回文件：{input_path}")

In [ ]:
sort_by_time("YOUR_PROJECT_DIR/dataset/mergebert/json_ts/autorest_typescript_time.json")
sort_by_time("YOUR_PROJECT_DIR/dataset/mergebert/json_ts/ssr_time.json")


In [22]:
sort_by_time("YOUR_PROJECT_DIR/dataset/mergebert/json_ts/autorest_typescript_time.json")
sort_by_time("YOUR_PROJECT_DIR/dataset/mergebert/json_ts/ssr_time.json")
sort_by_time("YOUR_PROJECT_DIR/dataset/mergebert/json_ts/stratos_time.json")
sort_by_time("YOUR_PROJECT_DIR/dataset/mergebert/json_ts/tv-automation-server-core_time.json")

已按时间从晚到早排序并写回文件：YOUR_PROJECT_DIR/dataset/mergebert/json_ts/autorest_typescript_time.json
已按时间从晚到早排序并写回文件：YOUR_PROJECT_DIR/dataset/mergebert/json_ts/ssr_time.json
已按时间从晚到早排序并写回文件：YOUR_PROJECT_DIR/dataset/mergebert/json_ts/stratos_time.json
已按时间从晚到早排序并写回文件：YOUR_PROJECT_DIR/dataset/mergebert/json_ts/tv-automation-server-core_time.json


## 分割前20%的数据

In [35]:
import json
import os


def split_json(source_file, output_40_file, output_60_file):
    # 读取源文件
    print(f"正在读取文件: {source_file}")
    with open(source_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # 统计数据数量
    total_count = len(data)
    print(f"总数据量: {total_count}")

    # 计算分割点
    split_index = int(total_count * 0.8)
    print(f"前40%数据量: {split_index}")
    print(f"后60%数据量: {total_count - split_index}")

    # 分割数据
    data_40 = data[:split_index]
    data_60 = data[split_index:]

    # 确保输出目录存在
    os.makedirs(os.path.dirname(output_40_file), exist_ok=True)
    os.makedirs(os.path.dirname(output_60_file), exist_ok=True)

    # 保存前40%的数据
    print(f"\n保存前40%数据到: {output_40_file}")
    with open(output_40_file, 'w', encoding='utf-8') as f:
        json.dump(data_40, f, ensure_ascii=False, indent=2)

    # 保存后40%的数据
    print(f"保存后80%数据到: {output_60_file}")
    with open(output_60_file, 'w', encoding='utf-8') as f:
        json.dump(data_60, f, ensure_ascii=False, indent=2)

    print("\n数据处理完成！")
    print(f"- {output_40_file}: {len(data_40)} 条数据")
    print(f"- {output_60_file}: {len(data_60)} 条数据")



In [24]:
split_json("YOUR_PROJECT_DIR/dataset/mergebert/json_ts/autorest_typescript_time.json",
           "YOUR_PROJECT_DIR/dataset/mergebert_ts_20/autorest_typescript_time20.json",
           "YOUR_PROJECT_DIR/dataset/mergebert_ts_80/autorest_typescript_time80.json")
split_json("YOUR_PROJECT_DIR/dataset/mergebert/json_ts/ssr_time.json",
           "YOUR_PROJECT_DIR/dataset/mergebert_ts_20/ssr_time20.json",
           "YOUR_PROJECT_DIR/dataset/mergebert_ts_80/ssr_time80.json")
split_json("YOUR_PROJECT_DIR/dataset/mergebert/json_ts/stratos_time.json",
           "YOUR_PROJECT_DIR/dataset/mergebert_ts_20/stratos_time20.json",
           "YOUR_PROJECT_DIR/dataset/mergebert_ts_80/stratos_time80.json")
split_json("YOUR_PROJECT_DIR/dataset/mergebert/json_ts/tv-automation-server-core_time.json",
           "YOUR_PROJECT_DIR/dataset/mergebert_ts_20/tv-automation-server-core_time20.json",
           "YOUR_PROJECT_DIR/dataset/mergebert_ts_80/tv-automation-server-core_time80.json")

正在读取文件: YOUR_PROJECT_DIR/dataset/mergebert/json_ts/autorest_typescript_time.json
总数据量: 745
前20%数据量: 149
后80%数据量: 596

保存前20%数据到: YOUR_PROJECT_DIR/dataset/mergebert_ts_20/autorest_typescript_time20.json
保存后80%数据到: YOUR_PROJECT_DIR/dataset/mergebert_ts_80/autorest_typescript_time80.json

数据处理完成！
- YOUR_PROJECT_DIR/dataset/mergebert_ts_20/autorest_typescript_time20.json: 149 条数据
- YOUR_PROJECT_DIR/dataset/mergebert_ts_80/autorest_typescript_time80.json: 596 条数据
正在读取文件: YOUR_PROJECT_DIR/dataset/mergebert/json_ts/ssr_time.json
总数据量: 633
前20%数据量: 126
后80%数据量: 507

保存前20%数据到: YOUR_PROJECT_DIR/dataset/mergebert_ts_20/ssr_time20.json
保存后80%数据到: YOUR_PROJECT_DIR/dataset/mergebert_ts_80/ssr_time80.json

数据处理完成！
- YOUR_PROJECT_DIR/dataset/mergebert_ts_20/ssr_time20.json: 126 条数据
- YOUR_PROJECT_DIR/dataset/mergebert_ts_80/ssr_time80.json: 507 条数据
正在读取文件: YOUR_PROJECT_DIR/dataset/mergebert/json_ts/stratos_time.json
总数据量: 419
前20%数据量: 83
后80%数据量: 336

保存前20%数据到: YOUR_PROJECT_DIR/dataset/mergebert_ts_

In [36]:

split_json("YOUR_PROJECT_DIR/dataset/mergebert/json/autopsy_time.json",
           "YOUR_PROJECT_DIR/dataset/mergebert_80/autopsy_time80.json",
           "YOUR_PROJECT_DIR/dataset/after/mergebert_20/autopsy_time20.json")

split_json("YOUR_PROJECT_DIR/dataset/mergebert/json/autorest_time.json",
           "YOUR_PROJECT_DIR/dataset/mergebert_80/autorest_time80.json",
           "YOUR_PROJECT_DIR/dataset/after/mergebert_20/autorest_time20.json")

split_json("YOUR_PROJECT_DIR/dataset/mergebert/json/azure_time.json",
           "YOUR_PROJECT_DIR/dataset/mergebert_80/azure_time80.json",
           "YOUR_PROJECT_DIR/dataset/after/mergebert_20/azure_time20.json")

split_json("YOUR_PROJECT_DIR/dataset/mergebert/json/cloudstack_time.json",
           "YOUR_PROJECT_DIR/dataset/mergebert_80/cloudstack_time80.json",
           "YOUR_PROJECT_DIR/dataset/after/mergebert_20/cloudstack_time20.json")

split_json("YOUR_PROJECT_DIR/dataset/mergebert/json/Galacticraft_time.json",
           "YOUR_PROJECT_DIR/dataset/mergebert_80/Galacticraft_time80.json",
           "YOUR_PROJECT_DIR/dataset/after/mergebert_20/Galacticraft_time20.json")

split_json("YOUR_PROJECT_DIR/dataset/mergebert/json/molgenis_time.json",
           "YOUR_PROJECT_DIR/dataset/mergebert_80/molgenis_time80.json",
           "YOUR_PROJECT_DIR/dataset/after/mergebert_20/molgenis_time20.json")

split_json("YOUR_PROJECT_DIR/dataset/mergebert/json/orientdb_time.json",
           "YOUR_PROJECT_DIR/dataset/mergebert_80/orientdb_time80.json",
           "YOUR_PROJECT_DIR/dataset/after/mergebert_20/orientdb_time20.json")

split_json("YOUR_PROJECT_DIR/dataset/mergebert/json/rumble_time.json",
           "YOUR_PROJECT_DIR/dataset/mergebert_80/rumble_time80.json",
           "YOUR_PROJECT_DIR/dataset/after/mergebert_20/rumble_time20.json")
           
split_json("YOUR_PROJECT_DIR/dataset/mergebert/json/sis_time.json",
           "YOUR_PROJECT_DIR/dataset/mergebert_80/sis_time80.json",
           "YOUR_PROJECT_DIR/dataset/after/mergebert_20/sis_time20.json")

split_json("YOUR_PROJECT_DIR/dataset/mergebert/json/SOS_time.json",
           "YOUR_PROJECT_DIR/dataset/mergebert_80/SOS_time80.json",
           "YOUR_PROJECT_DIR/dataset/after/mergebert_20/SOS_time20.json")

split_json("YOUR_PROJECT_DIR/dataset/mergebert/json/spring_time.json",
           "YOUR_PROJECT_DIR/dataset/mergebert_80/spring-boot_time80.json",
           "YOUR_PROJECT_DIR/dataset/after/mergebert_20/spring-boot_time20.json")

正在读取文件: YOUR_PROJECT_DIR/dataset/mergebert/json/autopsy_time.json
总数据量: 700
前40%数据量: 560
后60%数据量: 140

保存前40%数据到: YOUR_PROJECT_DIR/dataset/mergebert_80/autopsy_time80.json
保存后80%数据到: YOUR_PROJECT_DIR/dataset/after/mergebert_20/autopsy_time20.json

数据处理完成！
- YOUR_PROJECT_DIR/dataset/mergebert_80/autopsy_time80.json: 560 条数据
- YOUR_PROJECT_DIR/dataset/after/mergebert_20/autopsy_time20.json: 140 条数据
正在读取文件: YOUR_PROJECT_DIR/dataset/mergebert/json/autorest_time.json
总数据量: 864
前40%数据量: 691
后60%数据量: 173

保存前40%数据到: YOUR_PROJECT_DIR/dataset/mergebert_80/autorest_time80.json
保存后80%数据到: YOUR_PROJECT_DIR/dataset/after/mergebert_20/autorest_time20.json

数据处理完成！
- YOUR_PROJECT_DIR/dataset/mergebert_80/autorest_time80.json: 691 条数据
- YOUR_PROJECT_DIR/dataset/after/mergebert_20/autorest_time20.json: 173 条数据
正在读取文件: YOUR_PROJECT_DIR/dataset/mergebert/json/azure_time.json
总数据量: 927
前40%数据量: 741
后60%数据量: 186

保存前40%数据到: YOUR_PROJECT_DIR/dataset/mergebert_80/azure_time80.json
保存后80%数据到: YOUR_PROJECT_DIR

## 被阈值阻拦的数据使用nothing

## 观察BM25和L2的实际分布值

In [ ]:
import json
import torch
from pymilvus import MilvusClient, AnnSearchRequest, RRFRanker
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

CLUSTER_ENDPOINT = "YOUR_MILVUS_ENDPOINT"
TOKEN = "f294ac298b17866b1dc52fb0b7dff92ffcad2253e76041a41706d4b3de6621eb29e87782737717d43ebbcb58136f241b8dbdd01c"
COLL = "autopsy_time"
INPUT_JSON = r"YOUR_PROJECT_DIR\dataset\mergebert\json\autopsy_time.json"
OUTPUT_JSON = r"YOUR_PROJECT_DIR\dataset\mergebert\json\autopsy_time_tmp.json"

# 采样前多少条冲突用来探查分布
SAMPLE_N = 20
# 每路检索返回多少条
TOPK = 10

print("loading model/tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("./codeT5/codet5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("./codeT5/codet5-base").to("cuda" if torch.cuda.is_available() else "cpu")

print("loading data...")
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

print("connecting milvus...")
client = MilvusClient(uri=CLUSTER_ENDPOINT, token=TOKEN)

results = []
for idx, item in enumerate(data[:SAMPLE_N]):
    conflict = (
        "concurrent_change\n" + item.get("a_contents", "") +
        "\nbase_content\n" + item.get("o_contents", "") +
        "\nincoming_change\n" + item.get("b_contents", "")
    )
    query = conflict.replace("\n", " _newline_ ").replace("\t", " ")

    # 稀疏：BM25
    sparse_req = AnnSearchRequest([query], "chunk_sparse_embedding", {"metric_type": "BM25"}, limit=TOPK)
    sparse_res = client.hybrid_search(COLL, [sparse_req], ranker=RRFRanker(), limit=TOPK, output_fields=["global_id", "chunk_timestamp"])

    # 稠密：L2
    inputs = tokenizer(query, padding=True, truncation=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        enc = model.encoder(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        dense_vec = enc.last_hidden_state.mean(dim=1).cpu().numpy()[0]
    dense_req = AnnSearchRequest([dense_vec], "chunk_embedding", {"metric_type": "L2"}, limit=TOPK)
    dense_res = client.hybrid_search(COLL, [dense_req], ranker=RRFRanker(), limit=TOPK, output_fields=["global_id", "chunk_timestamp"])

    bm25_hits = [
        {
            "score": getattr(h, "score", None),
            "distance": getattr(h, "distance", None),
            "global_id": h["entity"].get("global_id"),
            "timestamp": h["entity"].get("chunk_timestamp"),
        }
        for h in sparse_res[0]
    ]
    l2_hits = [
        {
            "score": getattr(h, "score", None),
            "distance": getattr(h, "distance", None),
            "global_id": h["entity"].get("global_id"),
            "timestamp": h["entity"].get("chunk_timestamp"),
        }
        for h in dense_res[0]
    ]

    results.append({
        "query_idx": idx,
        "source_global_id": item.get("id"),
        "bm25_hits": bm25_hits,
        "l2_hits": l2_hits,
    })

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"done. saved {len(results)} queries to {OUTPUT_JSON}")

# mergeBert结果（基于时间历史）

## prompt1——动态扩展，370找不到相似

In [ ]:
import json
import random
from datetime import datetime
from utils import prompt1
from colorama import Fore, Style
from alibaba_api import qwen
from database_api_T5_BM25_L2 import query_similar

# 云端向量数据库配置
CLUSTER_ENDPOINT = "YOUR_MILVUS_ENDPOINT" # Serverless-T5-BM25-L2
TOKEN = "f294ac298b17866b1dc52fb0b7dff92ffcad2253e76041a41706d4b3de6621eb29e87782737717d43ebbcb58136f241b8dbdd01c"

def find_similar_conflicts_with_time_filter(repo_name, query, current_id, current_time, num_similar, initial_limit=10, max_limit=50, step=10):
    """
    动态查找相似冲突块，确保时间戳早于当前时间，且id不同。
    
    :param repo_name: 仓库名称
    :param query: 查询字符串
    :param current_id: 当前冲突块的id
    :param current_time: 当前冲突块的时间戳（datetime对象）
    :param num_similar: 需要查找的相似冲突数量
    :param initial_limit: 初始查询limit
    :param max_limit: 最大查询limit
    :param step: 每次增加的limit步长
    :return: 找到的冲突块列表 [(refer_c, refer_r), ...]，最多num_similar个
    """
    found_conflicts = []
    limit = initial_limit
    
    while len(found_conflicts) < num_similar and limit <= max_limit:
        # 查询相似冲突
        global_ids, related_cs, related_rs, timestamps = query_similar(CLUSTER_ENDPOINT, TOKEN, repo_name, query, limit)
        
        # 过滤时间戳晚于当前时间的记录
        if current_time:
            filtered_indices = [
                j for j, ts in enumerate(timestamps) 
                if ts and datetime.fromisoformat(ts) <= current_time
            ]
            global_ids = [global_ids[j] for j in filtered_indices]
            related_cs = [related_cs[j] for j in filtered_indices]
            related_rs = [related_rs[j] for j in filtered_indices]
        
        # 收集符合条件的冲突块（id不同）
        for j in range(len(global_ids)):
            if global_ids[j] != current_id and len(found_conflicts) < num_similar:
                refer_c = related_cs[j].replace(' _newline_ ', '\n').replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'
                refer_r = related_rs[j].replace('_newline_', '\n')
                found_conflicts.append((refer_c, refer_r))
                print(f"<<<<<<当前id: {current_id}, 相似冲突id: {global_ids[j]}>>>>>>")
        
        if len(found_conflicts) >= num_similar:
            break
        limit += step
    
    if len(found_conflicts) < num_similar:
        print(f"警告：即使扩展到limit={limit}，仍只找到 {len(found_conflicts)} 个符合条件的冲突块！")
    
    return found_conflicts

def request_gpt(input_path, output_path, num_similar):
    """
    处理JSON文件，为每个冲突块生成基于历史相似冲突的解决方案。
    
    :param input_path: 输入JSON文件路径
    :param output_path: 输出JSON文件路径
    :param num_similar: 需要查找的相似冲突数量，默认4
    """
    try:
        with open(input_path, 'r', encoding='utf-8') as file:
            json_data = json.load(file)
        
        # random.shuffle(json_data)  # 可选：随机打乱顺序
        
        for i, item in enumerate(json_data):
            # if i + 1 < 340:  # 从指定点继续运行
            #     continue
            # if i >= 500:  # 限制处理数量
            #     break
            
            current_id = item.get('id', '')
            repo_url = item.get('repo', '')
            repo_name = (repo_url.split('/')[-1].split('-')[0] + "_time") if repo_url else ''
            res_region = item.get('res_region', '')
            
            # 构建冲突字符串
            conflict = (
                'concurrent_change\n' + item.get('a_contents', '') +
                '\nbase_content\n' + item.get('o_contents', '') +
                '\nincoming_change\n' + item.get('b_contents', '')
            )
            query = conflict.replace('\n', ' _newline_ ').replace('\t', ' ')
            
            # 解析当前时间戳
            current_timestamp = item.get('timestamp', '')
            current_time = datetime.fromisoformat(current_timestamp) if current_timestamp else None
            
            # 查找相似冲突
            found_conflicts = find_similar_conflicts_with_time_filter(
                repo_name, query, current_id, current_time, num_similar=num_similar
            )
            
            # 填充refer变量（最多用前4个，因为prompt4固定）
            refer_c1, refer_r1 = found_conflicts[0] if len(found_conflicts) > 0 else ("", "")
            refer_c2, refer_r2 = found_conflicts[1] if len(found_conflicts) > 1 else ("", "")
            refer_c3, refer_r3 = found_conflicts[2] if len(found_conflicts) > 2 else ("", "")
            refer_c4, refer_r4 = found_conflicts[3] if len(found_conflicts) > 3 else ("", "")
            refer_c5, refer_r5 = found_conflicts[4] if len(found_conflicts) > 4 else ("", "")
            # 构建diff冲突字符串
            diff_conflict = conflict.replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'
            
            print(f"\n<--------------------------------------------示例 {i + 1} (line_withRefer{num_similar}_T5)-------------------------------------------->")
            item[f'line_withRefer{num_similar}_T5'] = qwen(
                question=prompt1(refer_c1, refer_r1, diff_conflict),
                number=5,
                truth=res_region
            )
    
    except Exception as e:
        print(f"{Fore.RED}Error----: 发生错误: {e}{Style.RESET_ALL}")
    finally:
        with open(output_path, 'w', encoding='utf-8') as file:
            json.dump(json_data, file, indent=2, ensure_ascii=False)
        print(f"{Fore.GREEN}Warning----: 数据已保存! 异常发生点: 第 {i + 1} 示例{Style.RESET_ALL}")

# 调用函数
request_gpt(
    input_path="YOUR_PROJECT_DIR/dataset/mergebert/json/autopsy_time.json",
    output_path="YOUR_PROJECT_DIR/produce/mergebert/time/autopsy_time.json",
    num_similar=1  # 可灵活设置为1-5
) 

## prompt1 query_similar改版后

In [ ]:
import json
import random
from datetime import datetime
from utils import simplePrompt, prompt1
from colorama import Fore, Style
from alibaba_api import qwen
from database_api_T5_BM25_L2 import query_similar

CLUSTER_ENDPOINT = "YOUR_MILVUS_ENDPOINT" # Serverless-T5-BM25-L2
TOKEN = "f294ac298b17866b1dc52fb0b7dff92ffcad2253e76041a41706d4b3de6621eb29e87782737717d43ebbcb58136f241b8dbdd01c"

def request_gpt(input, output):
    try:
        with open(input, 'r', encoding='utf-8') as input_file:
            json_data = json.load(input_file)
        
        # random.shuffle(json_data) # 随机洗牌顺序打开与否
        for i, item in enumerate(json_data):
            if i + 1 < 351: # 从异常发生点接着运行
                continue
            if i == 500:
                break

            id = item.get('id', '')
            repo_url = item.get('repo', '')
            repo_name = (repo_url.split('/')[-1].split('-')[0] + "_time") if repo_url else ''
            res_region = item.get('res_region', '')
            
            conflict = 'concurrent_change\n' + item.get('a_contents', '') + '\nbase_content\n' + item.get('o_contents', '') + '\nincoming_change\n' + item.get('b_contents', '')

            query = conflict.replace('\n', ' _newline_ ').replace('\t', ' ')
            current_timestamp = item.get('timestamp', '')
            current_ts = current_timestamp if current_timestamp else None
            # query_similar 现返回 4 个值：id、冲突、resolution、时间戳
            global_id, related_c, related_r, timestamps = query_similar(CLUSTER_ENDPOINT, TOKEN, repo_name, query, 3, current_ts=current_ts)
            diff_conflict = conflict.replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'

            for j in range(len(global_id)):  # 找到第一个 != id 的索引
                if global_id[j] != id:
                    print(f"<<<<<<我的id是{id}, 相似冲突id是{global_id[j]}>>>>>>")
                    refer_c1 = related_c[j].replace(' _newline_ ', '\n').replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'
                    refer_r1 = related_r[j].replace('_newline_', '\n')
                    break

            print(f"\n<--------------------------------------------示例{i + 1}(line_no)：-------------------------------------------->")
            item['line_no'] = qwen(question=simplePrompt(diff_conflict), number=3, truth=res_region)

            print(f"\n<--------------------------------------------示例{i + 1}(line_withRefer)：-------------------------------------------->")
            item['line_withRefer1'] = qwen(question=prompt1(refer_c1, refer_r1, diff_conflict), number=5, truth=res_region)

    except Exception as e:
        print(f"{Fore.RED}Error----:An error occurred: {e}{Style.RESET_ALL}")
    finally:
        with open(output, 'w', encoding='utf-8') as output_file:
            json.dump(json_data, output_file, indent=2)
        print(f"{Fore.GREEN}Warnning----:ChatGPT已运行数据已保存! 异常发生点:第{i + 1}示例{Style.RESET_ALL}")
        output_file.close()
request_gpt(input = "YOUR_PROJECT_DIR/produce/mergebert/time/autopsy_timev0.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert/time/autopsy_timev0.json") 
# request_gpt(input = "YOUR_PROJECT_DIR/produce/mergebert/time/autopsy_time.json", # 处理异常的时候这里也要改成相同的文件名
#             output= "YOUR_PROJECT_DIR/produce/mergebert/time/autopsy_time.json") 

## 额外置阈值的情况v2

In [ ]:
import json
import random
from datetime import datetime
from utils import simplePrompt, prompt1
from colorama import Fore, Style
from alibaba_api import qwen
from database_api_T5_BM25_L2_threshold import query_similar

CLUSTER_ENDPOINT = "YOUR_MILVUS_ENDPOINT" # Serverless-T5-BM25-L2
TOKEN = "f294ac298b17866b1dc52fb0b7dff92ffcad2253e76041a41706d4b3de6621eb29e87782737717d43ebbcb58136f241b8dbdd01c"

def request_gpt(input, output):
    try:
        with open(input, 'r', encoding='utf-8') as input_file:
            json_data = json.load(input_file)
        
        # random.shuffle(json_data) # 随机洗牌顺序打开与否
        for i, item in enumerate(json_data):
            # if i + 1 < 352: # 从异常发生点接着运行
            #     continue
            # if i == 500:  # 先测试前100条
            #     break

            id = item.get('id', '')
            repo_url = item.get('repo', '')
            repo_name = (repo_url.split('/')[-1].split('-')[0] + "_time") if repo_url else ''
            res_region = item.get('res_region', '')
            
            conflict = 'concurrent_change\n' + item.get('a_contents', '') + '\nbase_content\n' + item.get('o_contents', '') + '\nincoming_change\n' + item.get('b_contents', '')

            query = conflict.replace('\n', ' _newline_ ').replace('\t', ' ')
            current_timestamp = item.get('timestamp', '')
            current_ts = current_timestamp if current_timestamp else None
            
            # query_similar 现返回 7 个值：id、冲突、resolution、时间戳、RRF分值、BM25分值、L2距离
            global_id, related_c, related_r, timestamps, rrf_scores, bm25_scores, l2_distances = query_similar(
                CLUSTER_ENDPOINT, TOKEN, repo_name, query, 10, 
                current_ts=current_ts,
                enable_threshold=True,      # 启用阈值过滤
                bm25_threshold=0.0,        # BM25 最低分值（可调整）
                l2_threshold=2.5,           # L2 最大距离（可调整）
                rrf_threshold=0.0         # RRF 最低分值（可调整）
            )
            
            diff_conflict = conflict.replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'

            # 找到第一个 != id 的相似冲突
            refer_c1, refer_r1 = "", ""
            for j in range(len(global_id)):
                if global_id[j] != id:
                    print(f"<<<<<<我的id是{id}, 相似冲突id是{global_id[j]}, RRF={rrf_scores[j]:.4f}, BM25={bm25_scores[j]:.4f}, L2={l2_distances[j]:.4f}>>>>>>")
                    refer_c1 = related_c[j].replace(' _newline_ ', '\n').replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'
                    refer_r1 = related_r[j].replace('_newline_', '\n')
                    break
            
            # 如果没有找到相似冲突，跳过该样本或使用空参考
            if not refer_c1:
                print(f"警告: 样本 {i+1} (id={id}) 未找到符合阈值的相似冲突")

            # print(f"\n<--------------------------------------------示例{i + 1}(line_no)：-------------------------------------------->")
            # item['line_no'] = qwen(question=simplePrompt(diff_conflict), number=3, truth=res_region)

            if refer_c1:  # 只有在有参考时才调用
                print(f"\n<--------------------------------------------示例{i + 1}(line_withRefer)：-------------------------------------------->")
                item['line_withRefer1'] = qwen(question=prompt1(refer_c1, refer_r1, diff_conflict), number=5, truth=res_region)

    except Exception as e:
        print(f"{Fore.RED}Error----:An error occurred: {e}{Style.RESET_ALL}")
    finally:
        with open(output, 'w', encoding='utf-8') as output_file:
            json.dump(json_data, output_file, indent=2)
        print(f"{Fore.GREEN}Warnning----:ChatGPT已运行数据已保存! 异常发生点:第{i + 1}示例{Style.RESET_ALL}")
        output_file.close()
# request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/autorest_time20.json", # 处理异常的时候这里也要改成相同的文件名
#             output= "YOUR_PROJECT_DIR/produce/mergebert_20/autorest_time20v2.json") 
# request_gpt(input = "YOUR_PROJECT_DIR/produce/mergebert/time/autopsy_time_threshold.json", 
#             output= "YOUR_PROJECT_DIR/produce/mergebert/time/autopsy_time_threshold.json")

In [ ]:
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/autopsy_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_20/autopsy_time20v2.json") 

In [ ]:
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/autorest_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_20/autorest_time20v2.json") 

In [ ]:
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/azure_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_20/azure_time20v2.json") 
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/cloudstack_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_20/cloudstack_time20v2.json") 
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/Galacticraft_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_20/Galacticraft_time20v2.json") 
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/molgenis_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_20/molgenis_time20v2.json")
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/orientdb_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_20/orientdb_time20v2.json")
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/rumble_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_20/rumble_time20v2.json")
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/sis_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_20/sis_time20v2.json")
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/SOS_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_20/SOS_time20v2.json")
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/spring_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_20/spring_time20v2.json")

## mergebert整后80%的数据阈值2.5

In [ ]:
import json
import random
from datetime import datetime
from utils import simplePrompt, prompt1
from colorama import Fore, Style
from alibaba_api import qwen
from database_api_T5_BM25_L2_dense_only import query_similar

CLUSTER_ENDPOINT = "YOUR_MILVUS_ENDPOINT" # Serverless-T5-BM25-L2
TOKEN = "f294ac298b17866b1dc52fb0b7dff92ffcad2253e76041a41706d4b3de6621eb29e87782737717d43ebbcb58136f241b8dbdd01c"

def request_gpt(input, output):
    try:
        with open(input, 'r', encoding='utf-8') as input_file:
            json_data = json.load(input_file)
        
        # random.shuffle(json_data) # 随机洗牌顺序打开与否
        for i, item in enumerate(json_data):
            # if i + 1 < 352: # 从异常发生点接着运行
            #     continue
            # if i == 500:  # 先测试前100条
            #     break

            id = item.get('id', '')
            repo_name = "mergebert_80"
            res_region = item.get('res_region', '')
            
            conflict = 'concurrent_change\n' + item.get('a_contents', '') + '\nbase_content\n' + item.get('o_contents', '') + '\nincoming_change\n' + item.get('b_contents', '')

            query = conflict.replace('\n', ' _newline_ ').replace('\t', ' ')
            current_timestamp = item.get('timestamp', '')
            current_ts = current_timestamp if current_timestamp else None
            
            # query_similar 现返回 7 个值：id、冲突、resolution、时间戳、RRF分值、BM25分值、L2距离
            global_id, related_c, related_r, timestamps, rrf_scores, bm25_scores, l2_distances = query_similar(
                CLUSTER_ENDPOINT, TOKEN, repo_name, query, 10, 
                current_ts=None,
                enable_threshold=True,      # 启用阈值过滤
                bm25_threshold=0.0,        # BM25 最低分值（可调整）
                l2_threshold=2.5,           # L2 最大距离（可调整）
                rrf_threshold=0.0         # RRF 最低分值（可调整）
            )
            
            diff_conflict = conflict.replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'

            # 找到第一个 != id 的相似冲突
            refer_c1, refer_r1 = "", ""
            for j in range(len(global_id)):
                if global_id[j] != id:
                    print(f"<<<<<<我的id是{id}, 相似冲突id是{global_id[j]}, RRF={rrf_scores[j]:.4f}, BM25={bm25_scores[j]:.4f}, L2={l2_distances[j]:.4f}>>>>>>")
                    refer_c1 = related_c[j].replace(' _newline_ ', '\n').replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'
                    refer_r1 = related_r[j].replace('_newline_', '\n')
                    break
            
            # 如果没有找到相似冲突，跳过该样本或使用空参考
            if not refer_c1:
                print(f"警告: 样本 {i+1} (id={id}) 未找到符合阈值的相似冲突")

            # print(f"\n<--------------------------------------------示例{i + 1}(line_no)：-------------------------------------------->")
            # item['line_no'] = qwen(question=simplePrompt(diff_conflict), number=3, truth=res_region)

            if refer_c1:  # 只有在有参考时才调用
                print(f"\n<--------------------------------------------示例{i + 1}(line_withRefer)：-------------------------------------------->")
                item['line_withRefer1'] = qwen(question=prompt1(refer_c1, refer_r1, diff_conflict), number=5, truth=res_region)

    except Exception as e:
        print(f"{Fore.RED}Error----:An error occurred: {e}{Style.RESET_ALL}")
    finally:
        with open(output, 'w', encoding='utf-8') as output_file:
            json.dump(json_data, output_file, indent=2)
        print(f"{Fore.GREEN}Warnning----:ChatGPT已运行数据已保存! 异常发生点:第{i + 1}示例{Style.RESET_ALL}")
        output_file.close()


In [ ]:
# request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/autopsy_time20.json", # 处理异常的时候这里也要改成相同的文件名
#             output= "YOUR_PROJECT_DIR/produce/mergebert_20/autopsy_time20v10.json") 
# request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/autorest_time20.json", # 处理异常的时候这里也要改成相同的文件名
#             output= "YOUR_PROJECT_DIR/produce/mergebert_20/autorest_time20v10.json") 

# request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/azure_time20.json", # 处理异常的时候这里也要改成相同的文件名
#             output= "YOUR_PROJECT_DIR/produce/mergebert_20/azure_time20v10.json") 
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/cloudstack_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_20/cloudstack_time20v10.json") 
# request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/Galacticraft_time20.json", # 处理异常的时候这里也要改成相同的文件名
#             output= "YOUR_PROJECT_DIR/produce/mergebert_20/Galacticraft_time20v10.json") 
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/molgenis_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_20/molgenis_time20v10.json")
# request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/orientdb_time20.json", # 处理异常的时候这里也要改成相同的文件名
#             output= "YOUR_PROJECT_DIR/produce/mergebert_20/orientdb_time20v10.json")
# request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/rumble_time20.json", # 处理异常的时候这里也要改成相同的文件名
#             output= "YOUR_PROJECT_DIR/produce/mergebert_20/rumble_time20v10.json")
# request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/sis_time20.json", # 处理异常的时候这里也要改成相同的文件名
#             output= "YOUR_PROJECT_DIR/produce/mergebert_20/sis_time20v10.json")
# request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/SOS_time20.json", # 处理异常的时候这里也要改成相同的文件名
#             output= "YOUR_PROJECT_DIR/produce/mergebert_20/SOS_time20v10.json")
# request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/spring_time20.json", # 处理异常的时候这里也要改成相同的文件名
#             output= "YOUR_PROJECT_DIR/produce/mergebert_20/spring_time20v10.json")

In [7]:
# VH3
import json
import random
from utils import simplePrompt, prompt3
from colorama import Fore, Style
from openai_api import gpt_35
from database_api_T5_BM25_L2_threshold import query_similar

CLUSTER_ENDPOINT = "YOUR_MILVUS_ENDPOINT" # Serverless-T5-BM25-L2
TOKEN = "f294ac298b17866b1dc52fb0b7dff92ffcad2253e76041a41706d4b3de6621eb29e87782737717d43ebbcb58136f241b8dbdd01c"

def find_conflicts3(id, global_id, related_c, related_r):
    """找到 global_id 中前三个不等于指定 id 的索引对应的值。 """
    refer_c1, refer_r1, refer_c2, refer_r2, refer_c3, refer_r3 = "", "", "", "", "", ""
    found_count = 0  # 记录找到的符合条件的数量

    for j in range(len(global_id)):
        if global_id[j] != id:
            print(f"<<<<<<我的id是{id}, 相似冲突id是{global_id[j]}>>>>>>")
            refer_c = related_c[j].replace(' _newline_ ', '\n').replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'
            refer_r = related_r[j].replace('_newline_', '\n')

            if found_count == 0:
                refer_c1, refer_r1 = refer_c, refer_r
            elif found_count == 1:
                refer_c2, refer_r2 = refer_c, refer_r
            elif found_count == 2:
                refer_c3, refer_r3 = refer_c, refer_r
            found_count += 1
            if found_count == 3:  # 找到三个后停止
                break
    if found_count < 3:
        print("警告：未找到足够的符合条件的值！")
    return refer_c1, refer_r1, refer_c2, refer_r2, refer_c3, refer_r3

def request_gpt(input, output):
    try:
        with open(input, 'r', encoding='utf-8') as input_file:
            json_data = json.load(input_file)
        
        # random.shuffle(json_data) # 随机洗牌顺序打开与否
        for i, item in enumerate(json_data):
            # if i + 1 < 142: # 从异常发生点接着运行
            #     continue
            # if i == 500:
            #     break

            id = item.get('id', '')
            repo_name = "mergebert_80"
            res_region = item.get('res_region', '')
            
            conflict = 'concurrent_change\n' + item.get('a_contents', '') + '\nbase_content\n' + item.get('o_contents', '') + '\nincoming_change\n' + item.get('b_contents', '')

            query = conflict.replace('\n', ' _newline_ ').replace('\t', ' ')
            # query_similar 现返回 7 个值：id、冲突、resolution、时间戳、RRF分值、BM25分值、L2距离
            global_id, related_c, related_r, timestamps, rrf_scores, bm25_scores, l2_distances = query_similar(
                CLUSTER_ENDPOINT, TOKEN, repo_name, query, 4, 
                current_ts=None,
                enable_threshold=False,      # 启用阈值过滤
                bm25_threshold=0.0,        # BM25 最低分值（可调整）
                l2_threshold=float('inf'),           # L2 最大距离（可调整）
                rrf_threshold=0.0         # RRF 最低分值（可调整）
            )
            
            diff_conflict = conflict.replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'

            refer_c1, refer_r1, refer_c2, refer_r2, refer_c3, refer_r3 = find_conflicts3(id, global_id, related_c, related_r)

            print(f"\n<--------------------------------------------示例{i + 1}(line_no)：-------------------------------------------->")
            item['line_no'] = gpt_35(question=simplePrompt(diff_conflict), number=3, truth=res_region)

            # print(f"\n<--------------------------------------------示例{i + 1}(line_withRefer3)-------------------------------------------->")
            # item['line_withRefer3'] = gpt_4o(question=prompt3(refer_c1, refer_r1, refer_c2, refer_r2, refer_c3, refer_r3, diff_conflict), number=3, truth=res_region)

    except Exception as e:
        print(f"{Fore.RED}Error----:An error occurred: {e}{Style.RESET_ALL}")
    finally:
        with open(output, 'w', encoding='utf-8') as output_file:
            json.dump(json_data, output_file, indent=2)
        print(f"{Fore.GREEN}Warnning----:ChatGPT已运行数据已保存! 异常发生点:第{i + 1}示例{Style.RESET_ALL}")
        output_file.close()
    


In [8]:
# request_gpt(input = "YOUR_PROJECT_DIR/produce/mergebert_20/autopsy_time20vno.json", # 处理异常的时候这里也要改成相同的文件名
#             output= "YOUR_PROJECT_DIR/produce/mergebert_20/autopsy_time20vno.json") 
# request_gpt(input = "YOUR_PROJECT_DIR/produce/mergebert_20/autorest_time20vno.json", # 处理异常的时候这里也要改成相同的文件名
#             output= "YOUR_PROJECT_DIR/produce/mergebert_20/autorest_time20vno.json") 

# request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/azure_time20.json", # 处理异常的时候这里也要改成相同的文件名
#             output= "YOUR_PROJECT_DIR/produce/mergebert_20/azure_time20vno.json") 
# request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/cloudstack_time20.json", # 处理异常的时候这里也要改成相同的文件名
#             output= "YOUR_PROJECT_DIR/produce/mergebert_20/cloudstack_time20vno.json") 
# request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/Galacticraft_time20.json", # 处理异常的时候这里也要改成相同的文件名
#             output= "YOUR_PROJECT_DIR/produce/mergebert_20/Galacticraft_time20vno.json") 
# request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/molgenis_time20.json", # 处理异常的时候这里也要改成相同的文件名
#             output= "YOUR_PROJECT_DIR/produce/mergebert_20/molgenis_time20vno.json")

# request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/orientdb_time20.json", # 处理异常的时候这里也要改成相同的文件名
#             output= "YOUR_PROJECT_DIR/produce/mergebert_20/orientdb_time20vno.json")
# request_gpt(input = "YOUR_PROJECT_DIR/produce/mergebert_20/rumble_time20vno.json", # 处理异常的时候这里也要改成相同的文件名
#             output= "YOUR_PROJECT_DIR/produce/mergebert_20/rumble_time20vno.json")

request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/sis_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_20/sis_time20vno.json")
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/SOS_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_20/SOS_time20vno.json")
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/spring_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_20/spring_time20vno.json")

使用本地模型
检索到 4 个结果（启用阈值过滤: False）
<<<<<<我的id是14073, 相似冲突id是14442>>>>>>
<<<<<<我的id是14073, 相似冲突id是14288>>>>>>
<<<<<<我的id是14073, 相似冲突id是14207>>>>>>

<--------------------------------------------示例1(line_no)：-------------------------------------------->

Task Instructions: Your task is to resolve the given conflict blocks by leveraging extensive experience in resolving code merge conflict.
Merge Conflict to be resolved Next: The merge Conflict block is given below - 
<<<<<<<
 * @version 1.0

|||||||
 * @version 1.0
 *
 * @see org.opengis.metadata.content.SampleDimension
 *


 * @version 1.1
 *
 * @see org.opengis.metadata.content.SampleDimension
 *

>>>>>>>.
Resolve the "Merge Conflict to Be Resolved Next" and produce the "Resolved Code" below according to the "Task Instructions". Please enclose the output code with markdown ```.


<Truth Resolution>:-------->
 * @version 1.1

</Truth Resolution>-------->

---------------ChatGPT的第1个Resolution---------------
```java
 * @version 1.1
 *
 * @see

In [43]:
import json
import random
from utils import simplePrompt, prompt1
from colorama import Fore, Style
from openai_api import gpt_35
from database_api_T5_BM25_L2_threshold import query_similar

CLUSTER_ENDPOINT = "YOUR_MILVUS_ENDPOINT" # Serverless-T5-BM25-L2
TOKEN = "f294ac298b17866b1dc52fb0b7dff92ffcad2253e76041a41706d4b3de6621eb29e87782737717d43ebbcb58136f241b8dbdd01c"

def request_gpt(input, output):
    try:
        with open(input, 'r', encoding='utf-8') as input_file:
            json_data = json.load(input_file)
        
        # random.shuffle(json_data) # 随机洗牌顺序打开与否
        for i, item in enumerate(json_data):
            # if i + 1 < 352: # 从异常发生点接着运行
            #     continue
            # if i == 500:  # 先测试前100条
            #     break

            id = item.get('id', '')
            repo_name = "mergebert_20"
            res_region = item.get('res_region', '')
            
            conflict = 'concurrent_change\n' + item.get('a_contents', '') + '\nbase_content\n' + item.get('o_contents', '') + '\nincoming_change\n' + item.get('b_contents', '')

            query = conflict.replace('\n', ' _newline_ ').replace('\t', ' ')
            current_timestamp = item.get('timestamp', '')
            current_ts = current_timestamp if current_timestamp else None
            
            # query_similar 现返回 7 个值：id、冲突、resolution、时间戳、RRF分值、BM25分值、L2距离
            global_id, related_c, related_r, timestamps, rrf_scores, bm25_scores, l2_distances = query_similar(
                CLUSTER_ENDPOINT, TOKEN, repo_name, query, 10, 
                current_ts=None,
                enable_threshold=False,      # 启用阈值过滤
                bm25_threshold=0.0,        # BM25 最低分值（可调整）
                l2_threshold=2.5,           # L2 最大距离（可调整）
                rrf_threshold=0.0         # RRF 最低分值（可调整）
            )
            
            diff_conflict = conflict.replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'

            # 找到第一个 != id 的相似冲突
            refer_c1, refer_r1 = "", ""
            for j in range(len(global_id)):
                if global_id[j] != id:
                    print(f"<<<<<<我的id是{id}, 相似冲突id是{global_id[j]}, RRF={rrf_scores[j]:.4f}, BM25={bm25_scores[j]:.4f}, L2={l2_distances[j]:.4f}>>>>>>")
                    refer_c1 = related_c[j].replace(' _newline_ ', '\n').replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'
                    refer_r1 = related_r[j].replace('_newline_', '\n')
                    break
            
            # 如果没有找到相似冲突，跳过该样本或使用空参考
            if not refer_c1:
                print(f"警告: 样本 {i+1} (id={id}) 未找到符合阈值的相似冲突")

            # print(f"\n<--------------------------------------------示例{i + 1}(line_no)：-------------------------------------------->")
            # item['line_no'] = gpt_35(question=simplePrompt(diff_conflict), number=3, truth=res_region)

            if refer_c1:  # 只有在有参考时才调用
                print(f"\n<--------------------------------------------示例{i + 1}(line_withRefer)：-------------------------------------------->")
                item['line_withRefer1'] = gpt_35(question=prompt1(refer_c1, refer_r1, diff_conflict), number=3, truth=res_region)

    except Exception as e:
        print(f"{Fore.RED}Error----:An error occurred: {e}{Style.RESET_ALL}")
    finally:
        with open(output, 'w', encoding='utf-8') as output_file:
            json.dump(json_data, output_file, indent=2)
        print(f"{Fore.GREEN}Warnning----:ChatGPT已运行数据已保存! 异常发生点:第{i + 1}示例{Style.RESET_ALL}")
        output_file.close()


In [45]:
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/autopsy_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_80/autopsy_time80.json") 
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/autorest_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_80/autorest_time80.json") 

request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/azure_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_80/azure_time80.json") 
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/cloudstack_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_80/cloudstack_time80.json") 
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/Galacticraft_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_80/Galacticraft_time80.json") 
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/molgenis_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_80/molgenis_time80.json")

request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/orientdb_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_80/orientdb_time80.json")
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/rumble_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_80/rumble_time80.json")

request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/sis_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_80/sis_time80.json")
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/SOS_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_80/SOS_time80.json")
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/spring_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_80/spring_time80.json")

使用本地模型
检索到 10 个结果（启用阈值过滤: False）
<<<<<<我的id是79837, 相似冲突id是79836, RRF=0.0325, BM25=140.5420, L2=2.2504>>>>>>

<--------------------------------------------示例1(line_withRefer)：-------------------------------------------->

Task Instructions: Your task is to resolve the given conflict blocks by leveraging extensive experience in resolving code merge conflict and the provided examples of merge conflict resolutions in the same code repository.
Reference example of merge conflict resolution in the same code repository: 
Reference 1. The similar merge conflict is - 
<<<<<<<
        bbattributes.add(new BlackboardAttribute(ATTRIBUTE_TYPE.TSK_MSG_ID.getTypeID(), EmailParserModuleFactory.getModuleName(), ((id < 0L) ? "Not available" : String.valueOf(id))));

|||||||
        bbattributes.add(new BlackboardAttribute(ATTRIBUTE_TYPE.TSK_MSG_ID.getTypeID(), MODULE_NAME, ((id < 0L) ? "Not available" : String.valueOf(id))));


        bbattributes.add(new BlackboardAttribute(ATTRIBUTE_TYPE.TSK_MSG_ID.g

2026-03-05 00:49:33,912 [ERROR][_create_connection]: Failed to create new connection using: 16bc4f0d04fd44d98b0b87b8b879862d (milvus_client.py:916)


Error----:An error occurred: <MilvusException: (code=2, message=Fail connecting to server on YOUR_MILVUS_ENDPOINT:443, illegal connection params or server unavailable)>
Warnning----:ChatGPT已运行数据已保存! 异常发生点:第39示例
使用本地模型
检索到 10 个结果（启用阈值过滤: False）
<<<<<<我的id是27441, 相似冲突id是79369, RRF=0.0320, BM25=38.0722, L2=1.3502>>>>>>

<--------------------------------------------示例1(line_withRefer)：-------------------------------------------->

Task Instructions: Your task is to resolve the given conflict blocks by leveraging extensive experience in resolving code merge conflict and the provided examples of merge conflict resolutions in the same code repository.
Reference example of merge conflict resolution in the same code repository: 
Reference 1. The similar merge conflict is - 
<<<<<<<

|||||||
import org.openide.util.Exceptions;


import org.openide.util.Exceptions;
import org.openide.util.NbBundle;

>>>>>>>
; and Reference 1 its resolution is - 
import org.openide.util.NbBundle; 
.
Merge Conflict

2026-03-05 01:20:15,146 [ERROR][_create_connection]: Failed to create new connection using: 5fc0b9bc7c2e46f49f86c29e6341c359 (milvus_client.py:916)


Error----:An error occurred: <MilvusException: (code=2, message=Fail connecting to server on YOUR_MILVUS_ENDPOINT:443, illegal connection params or server unavailable)>
Warnning----:ChatGPT已运行数据已保存! 异常发生点:第99示例
使用本地模型
检索到 10 个结果（启用阈值过滤: False）
<<<<<<我的id是466, 相似冲突id是52215, RRF=0.0164, BM25=48.2060, L2=inf>>>>>>

<--------------------------------------------示例1(line_withRefer)：-------------------------------------------->

Task Instructions: Your task is to resolve the given conflict blocks by leveraging extensive experience in resolving code merge conflict and the provided examples of merge conflict resolutions in the same code repository.
Reference example of merge conflict resolution in the same code repository: 
Reference 1. The similar merge conflict is - 
<<<<<<<
                    GalacticraftCore.packetPipeline.sendTo(new PacketSimple(EnumSimplePacket.C_OPEN_SPACE_RACE_GUI, player.worldObj.provider.getDimension(), new Object[] {}), player);

|||||||
                    Galactic

# mergeBert预处理（基于时间历史）

In [ ]:
import json
import os
from utils import rewrite_DiffMarks
from collections import defaultdict

# 需要筛选的 repo 列表
target_repos = [
    # "https://github.com/RumbleDB/rumble", # : 1415
    # "https://github.com/Azure/azure-sdk-for-java", # : 958
    # "https://github.com/52North/SOS", # : 902
    # "https://github.com/Azure/autorest", # : 866
    # "https://github.com/molgenis/molgenis", # : 797
    # "https://github.com/sleuthkit/autopsy", # : 775
    # "https://github.com/micdoodle8/Galacticraft" # : 737
    # "https://github.com/apache/sis", # : 598
    # "https://github.com/spring-projects/spring-boot", # : 586
    # "https://github.com/apache/cloudstack" # : 533
    "https://github.com/orientechnologies/orientdb" #: 529
]

def extract_and_save_data(json_file):
    repo_data = defaultdict(list)
    with open(json_file, 'r', encoding='utf-8') as file:
        data = json.load(file)
    
    for item in data:
        # 检查 res_label 是否为 "null"
        if item.get('res_region') == "null":
            continue

        # 计算总行数
        total_lines = (item.get("a_contents", "").count('\n') + 1) + (item.get("o_contents", "").count('\n') + 1) + (item.get("b_contents", "").count('\n') + 1)
        if total_lines > 36:
            continue

        if 'repo' in item and item['repo'] in target_repos:

            # 根据 json_name 打开对应的 JSON 文件获取 commitHash
            json_path = 'YOUR_DATA_DIR/mergebert_dataset/fse2022/automated-analysis-data/Java/' + item.get("json_name", "")
            commit_hash = ""
            if json_path and os.path.exists(json_path):
                try:
                    with open(json_path, 'r', encoding='utf-8') as json_file_content:
                        json_content = json.load(json_file_content)
                        commit_hash = json_content.get("commitHash", "")
                except Exception as e:
                    print(f"无法读取文件 {json_path}: {e}")
                    commit_hash = ""
            else:
                commit_hash = ""

            extracted_item = {
                "file_name": item.get("file_name", ""),
                "repo": item.get("repo", ""),
                "commitHash": commit_hash,
                "res_label": item.get("res_label", ""),
                "id": item.get("id", ""),
                "json_name": item.get("json_name", ""),
                "token_level_result": rewrite_DiffMarks(item.get("token_level_result", "")),
                "a_contents": item.get("a_contents", ""),
                "o_contents": item.get("o_contents", ""),
                "b_contents": item.get("b_contents", ""),
                "res_region": item.get("res_region", "")
            }
            repo_data[item['repo']].append(extracted_item)

    for repo_url, items in repo_data.items():
        repo_name = repo_url.split('/')[-1]
        output_file = f"dataset/mergebert/json/{repo_name}.json" # 写入 JSON 文件路径

        with open(output_file, 'w', encoding='utf-8') as outfile:
            json.dump(items, outfile, indent=4, ensure_ascii=False)
        
        print(f"已保存 {len(items)} 条数据到文件: {output_file}")

json_file = 'YOUR_DATA_DIR/mergeMinePython/json/mergeBert/javaContextPrettyVersionAll2.json'
extract_and_save_data(json_file)

# 50repo结果（基于时间历史）

In [ ]:
import json
import random
from datetime import datetime
from utils import simplePrompt, prompt1
from colorama import Fore, Style
from alibaba_api import qwen
from database_api_T5_BM25_L2 import query_similar

CLUSTER_ENDPOINT = "YOUR_MILVUS_ENDPOINT" # Serverless-T5-BM25-L2
TOKEN = "f294ac298b17866b1dc52fb0b7dff92ffcad2253e76041a41706d4b3de6621eb29e87782737717d43ebbcb58136f241b8dbdd01c"

def request_gpt(input, output):
    try:
        with open(input, 'r', encoding='utf-8') as input_file:
            json_data = json.load(input_file)
        
        # random.shuffle(json_data) # 随机洗牌顺序打开与否
        for i, item in enumerate(json_data):
            if i + 1 < 51: # 从异常发生点接着运行
                continue
            # if i == 500:
            #     break

            id = item.get('id', '')
            repo_url = item.get('repo', '')
            repo_name = (repo_url.split('/')[-1].split('-')[0] + "_time") if repo_url else ''
            res_region = item.get('res_region', '')
            
            conflict = 'concurrent_change\n' + item.get('a_contents', '') + '\nbase_content\n' + item.get('o_contents', '') + '\nincoming_change\n' + item.get('b_contents', '')

            query = conflict.replace('\n', ' _newline_ ').replace('\t', ' ')
            current_timestamp = item.get('timestamp', '')
            current_ts = current_timestamp if current_timestamp else None
            # query_similar 现返回 4 个值：id、冲突、resolution、时间戳
            global_id, related_c, related_r, timestamps = query_similar(CLUSTER_ENDPOINT, TOKEN, repo_name, query, 3, current_ts=current_ts)
            diff_conflict = conflict.replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'

            for j in range(len(global_id)):  # 找到第一个 != id 的索引
                if global_id[j] != id:
                    print(f"<<<<<<我的id是{id}, 相似冲突id是{global_id[j]}>>>>>>")
                    refer_c1 = related_c[j].replace(' _newline_ ', '\n').replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'
                    refer_r1 = related_r[j].replace('_newline_', '\n')
                    break

            print(f"\n<--------------------------------------------示例{i + 1}(line_no)：-------------------------------------------->")
            item['line_no'] = qwen(question=simplePrompt(diff_conflict), number=3, truth=res_region)

            print(f"\n<--------------------------------------------示例{i + 1}(line_withRefer)：-------------------------------------------->")
            item['line_withRefer1'] = qwen(question=prompt1(refer_c1, refer_r1, diff_conflict), number=5, truth=res_region)

    except Exception as e:
        print(f"{Fore.RED}Error----:An error occurred: {e}{Style.RESET_ALL}")
    finally:
        with open(output, 'w', encoding='utf-8') as output_file:
            json.dump(json_data, output_file, indent=2)
        print(f"{Fore.GREEN}Warnning----:ChatGPT已运行数据已保存! 异常发生点:第{i + 1}示例{Style.RESET_ALL}")
        output_file.close()

request_gpt(input = "YOUR_PROJECT_DIR/produce/mergebert/time/autopsy_time.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert/time/autopsy_time.json") 